# Clase 1 · Ingesta y capa Bronze

Vamos a generar cuatro fuentes, inspeccionar sus formatos y materializarlas como tablas Delta Bronze. Bronze conserva el dato recibido: los problemas se identifican, pero se corrigen recién en Silver.

In [0]:
dbutils.widgets.text("student_id", "narela_armani", "Identificador del alumno")
dbutils.widgets.dropdown("scale", "small", ["test", "small", "demo"], "Escala")

In [0]:
%run ../common/config

In [0]:
%run ../common/generate_data

In [0]:
catalog = spark.sql("SELECT current_catalog()").first()[0]
config = CourseConfig(catalog, dbutils.widgets.get("student_id"), dbutils.widgets.get("scale"))
create_course_namespace(spark, config)
data = build_course_data(spark, config)
paths = write_landing_files(data, config)
paths

{'customers_csv': '/Volumes/workspace/bigdata_narela_armani/landing/customers_csv',
 'products_parquet': '/Volumes/workspace/bigdata_narela_armani/landing/products_parquet',
 'transactions_csv': '/Volumes/workspace/bigdata_narela_armani/landing/transactions_csv',
 'events_json': '/Volumes/workspace/bigdata_narela_armani/landing/events_json'}

## 1. Leer no es todavía transformar

Inspeccioná los cuatro orígenes. Compará el esquema inferido del CSV con el esquema físico de Parquet. ¿Por qué `amount` termina como texto?

In [0]:
customers_raw = spark.read.option("header", True).csv(paths["customers_csv"])
transactions_raw = spark.read.option("header", True).csv(paths["transactions_csv"])
products_raw = spark.read.parquet(paths["products_parquet"])
events_raw = spark.read.json(paths["events_json"])

for name, frame in {"customers": customers_raw, "transactions": transactions_raw, "products": products_raw, "events": events_raw}.items():
    print(f"\n--- {name} ---")
    frame.printSchema()


--- customers ---
root
 |-- customer_id: string (nullable = true)
 |-- country: string (nullable = true)
 |-- segment: string (nullable = true)
 |-- created_date: string (nullable = true)
 |-- email: string (nullable = true)


--- transactions ---
root
 |-- transaction_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- event_ts: string (nullable = true)
 |-- amount: string (nullable = true)
 |-- payment_channel: string (nullable = true)
 |-- device_id: string (nullable = true)
 |-- is_fraud: string (nullable = true)


--- products ---
root
 |-- product_id: long (nullable = true)
 |-- category: string (nullable = true)
 |-- price: decimal(12,2) (nullable = true)


--- events ---
root
 |-- context: struct (nullable = true)
 |    |-- platform: string (nullable = true)
 |    |-- session_id: string (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- event_id: long (nullable = true)
 |-- event_ts: string (nullab

### Experimento: inferencia de tipos

Volvé a leer transacciones con `inferSchema=true`. Registrá cuánto tarda y verificá si resuelve correctamente `amount`. Explicá por qué inferir implica trabajo adicional y puede producir decisiones inestables.

In [0]:
transactions_inferred = (spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(paths["transactions_csv"]))
transactions_inferred.printSchema()
display(transactions_inferred.limit(10))

root
 |-- transaction_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- event_ts: timestamp (nullable = true)
 |-- amount: string (nullable = true)
 |-- payment_channel: string (nullable = true)
 |-- device_id: string (nullable = true)
 |-- is_fraud: integer (nullable = true)



transaction_id,customer_id,product_id,event_ts,amount,payment_channel,device_id,is_fraud
0,2049,497,2026-03-07T02:30:01.000Z,N/A,wallet,device_2102,0
1,3190,412,2026-03-02T05:17:39.000Z,408.25,wallet,device_1049,0
2,2223,340,2026-03-06T06:21:30.000Z,1515.54,wallet,device_2062,0
3,4951,292,2026-03-04T21:50:49.000Z,1533.05,card,device_859,0
4,1980,387,2026-03-06T10:42:23.000Z,56.14,transfer,device_225,0
5,4405,326,2026-03-04T22:49:51.000Z,969.70,transfer,device_1267,0
6,426,251,2026-03-03T19:48:32.000Z,1487.09,card,device_240,0
7,4371,8,2026-03-07T17:58:49.000Z,1140.12,transfer,device_1364,0
8,2572,176,2026-03-06T00:33:55.000Z,164.13,wallet,device_1799,0
9,763,354,2026-03-02T22:27:16.000Z,1523.97,transfer,device_267,0


## 2. Metadatos de ingesta

Bronze agrega trazabilidad sin modificar la semántica del dato de origen.

In [0]:
from pyspark.sql import functions as F

def with_ingestion_metadata(frame, source_name, source_path):
    return (frame
        .withColumn("_source", F.lit(source_name))
        .withColumn("_source_file", F.lit(source_path))
        .withColumn("_ingested_at", F.current_timestamp()))

bronze = {
    "customers": with_ingestion_metadata(customers_raw, "customers_csv", paths["customers_csv"]),
    "products": with_ingestion_metadata(products_raw, "products_parquet", paths["products_parquet"]),
    "transactions": with_ingestion_metadata(transactions_raw, "transactions_csv", paths["transactions_csv"]),
    "events": with_ingestion_metadata(events_raw, "events_json", paths["events_json"]),
}

In [0]:
for name, frame in bronze.items():
    table = f"{config.catalog}.{config.schema}.bronze_{name}"
    frame.write.format("delta").mode("overwrite").option("overwriteSchema", True).saveAsTable(table)
    print(f"{table}: {spark.table(table).count():,} filas")

workspace.bigdata_narela_armani.bronze_customers: 5,000 filas
workspace.bigdata_narela_armani.bronze_products: 500 filas
workspace.bigdata_narela_armani.bronze_transactions: 50,011 filas
workspace.bigdata_narela_armani.bronze_events: 200,000 filas


## 3. Diagnóstico inicial de calidad

No limpies todavía. Medí cuántos identificadores están duplicados y cuántos importes no pueden convertirse a número.

In [0]:
tx = spark.table(f"{config.catalog}.{config.schema}.bronze_transactions")
quality_summary = tx.agg(
    F.count("*").alias("rows"),
    F.countDistinct("transaction_id").alias("distinct_ids"),
    F.sum(F.expr("CASE WHEN try_cast(amount AS DECIMAL(12,2)) IS NULL THEN 1 ELSE 0 END")).alias("invalid_amounts"),
)
display(quality_summary)

rows,distinct_ids,invalid_amounts
50011,50000,52


## 4. Delta y el plan de ejecución

Ejecutá `DESCRIBE DETAIL`, `DESCRIBE HISTORY` y `EXPLAIN FORMATTED` sobre una tabla. Identificá qué información agrega Delta respecto de un directorio Parquet.

In [0]:
table_name = f"{config.catalog}.{config.schema}.bronze_transactions"
display(spark.sql(f"DESCRIBE DETAIL {table_name}"))
display(spark.sql(f"DESCRIBE HISTORY {table_name}"))
spark.sql(f"EXPLAIN FORMATTED SELECT payment_channel, count(*) FROM {table_name} GROUP BY payment_channel").show(truncate=False)

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,72572670-4f44-49e2-9d39-fab4d0a37f5f,workspace.bigdata_narela_armani.bronze_transactions,null,,2026-09-26T13:44:39.404Z,2026-09-26T13:44:42.000Z,List(),List(),1,687096,"Map(delta.parquet.compression.codec -> zstd, delta.parquet.format.version.afe.internal -> 2.12.0, delta.enableDeletionVectors -> true, delta.parquet.format.version -> 2.12.0, io.unitycatalog.tableId -> 42d374d1-d7ac-4674-a688-b8f0f9e4279b)",3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
0,2026-09-26T13:44:42.000Z,77567147740518,narmani@itba.edu.ar,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, canOverwriteSchema -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.enableDeletionVectors"":""true"",""delta.parquet.format.version"":""2.12.0"",""io.unitycatalog.tableId"":""42d374d1-d7ac-4674-a688-b8f0f9e4279b""}, statsOnLoad -> true)",null,List(3369095630238969),bcba8a9a-9aa3-4bd0-89dc-daef70f20c8a,0926-134318-bsc56dgi-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 50011, numOutputBytes -> 687096)",null,Databricks-Runtime/19.6.x-aarch64-photon-scala2.13


+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

## Entrega breve

Dejá ejecutadas en este notebook las salidas de las cuatro tablas y del diagnóstico de calidad. Las respuestas escritas van en el `README.md` de tu entrega: (1) tres observaciones sobre CSV/JSON, Parquet y Delta; y (2) una explicación de dónde aparece volumen, velocidad, variedad, veracidad y valor en este caso. El formato completo está en la sección **Formato de entrega** del README de la práctica.

## 1. Leer no es todavía transformar

**¿Por qué `amount` termina como texto?**
Porque el CSV no trae el tipo de dato de cada columna y nosotros tampoco lo especificamos. Sin esa info, Spark lee todo como `string` por default. El Parquet, en cambio, guarda el esquema dentro del archivo, por eso ahí los tipos aparecen bien.

**Experimento con `inferSchema`**
Aunque le pidamos que infiera, `amount` sigue quedando como texto, porque entre los valores hay `N/A`. Alcanza con un solo valor que no sea número para que Spark decida que toda la columna es `string`.

Inferir es trabajo adicional porque Spark tiene que recorrer los datos una vez más solo para adivinar los tipos antes de leerlos. Además, la decisión es inestable: depende de lo que venga en los datos. Si mañana llega un valor raro, el tipo puede cambiar. Lo mejor es definir el esquema nosotros y hacerlo bien de una.

## 3. Diagnóstico inicial de calidad

- Identificadores duplicados: **10**
- Importes que no se pueden convertir a número: **52**

## 4. Delta y el plan de ejecución

**¿Qué agrega Delta respecto de un directorio Parquet?**
- `DESCRIBE DETAIL` muestra la info de la tabla: el formato (`delta`), dónde está guardada, cuántos archivos tiene, cuánto pesa y cuándo se modificó por última vez.
- `DESCRIBE HISTORY` muestra cada versión de la tabla: qué operación se hizo, cuándo, quién la hizo y cuántas filas se escribieron.

Un directorio Parquet es solo una carpeta con archivos y no guarda nada de esto. Delta lo sabe porque tiene el `_delta_log`, un registro de todos los cambios que se le hicieron a la tabla.